## ALL 4 cells lines

In [1]:
import os
import pandas as pd
import numpy as np

df = pd.read_csv("/home/suncz/work/s02/Encode_epigenome/results/preprocessed/[241209]target_human.txt", sep="\t")
df.head()
df = df.query('notes.str.contains("K562|MCF-7|GM12878|HepG2", na=False)')


factor_celltype_count = df.groupby('factors')['notes'].nunique()
factors_in_multiple_celltypes = factor_celltype_count[factor_celltype_count >= 4]

factor_dict = {}
for index,name in enumerate(factors_in_multiple_celltypes.index.tolist()):
    factor_dict[name] = index


factors_order = factors_in_multiple_celltypes.index.tolist()

# Create a mapping dict to assign an order index to each factor
factor_position = {factor: i for i, factor in enumerate(factors_order)}

# Add a sort column
df['sort_order'] = df['factors'].map(factor_position)

# Sort by the sort column
df_sorted = df.sort_values('sort_order').drop('sort_order', axis=1)

df_sorted_unique = df_sorted.drop_duplicates(subset=['factors', 'notes'], keep='first')

In [2]:
sequence_bed = pd.read_csv("/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed/IMR-90_only/1m_akita_sequences.bed", sep="\t", header=None)
sequence_bed.columns = ['chrom', 'start', 'end', 'name']
train_index = sequence_bed[~sequence_bed["chrom"].isin(['chr2', 'chr10', 'chr21'])]
valid_index = sequence_bed[sequence_bed["chrom"].isin(['chr2'])]
test_index = sequence_bed[sequence_bed["chrom"].isin(['chr10', 'chr21'])]

In [3]:
BASE_DIR = "/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed"
source_dir = "/home/suncz/work/s02/Encode_epigenome/results/preprocessed/1m-akita/seqs_cov"

all_train_targets = []
all_valid_targets = []
all_test_targets = []

all_train_contigs = []
all_valid_contigs = []
all_test_contigs = []

for cell_type in ["K562", "MCF-7", "GM12878", "HepG2"]:
    df_selected = df_sorted_unique[(df_sorted_unique["notes"] == cell_type) & (df_sorted_unique["factors"].isin(factors_in_multiple_celltypes.index))]
    nums_ls = []
    index_ls = []
    for name in df_selected.iterrows():
        line = name[1]
        tfs = line['factors']
        nums = factor_dict[tfs]
        nums_ls.append(nums)
        index_ls.append(line['index'])

    target_ls = []
    for index in index_ls:
        source_h5 = os.path.join(source_dir, f"{index}.h5.npy")
        target = np.load(source_h5, allow_pickle=True)
        target_ls.append(target)
    targets = np.stack(target_ls,axis=-1)
    all_train_targets.append(targets[train_index.index,:,:])
    all_valid_targets.append(targets[valid_index.index,:,:])
    all_test_targets.append(targets[test_index.index,:,:])
    train_contigs = train_index.copy()
    train_contigs['cell_type'] = cell_type
    valid_contigs = valid_index.copy()
    valid_contigs['cell_type'] = cell_type
    test_contigs = test_index.copy()
    test_contigs['cell_type'] = cell_type
    all_train_contigs.append(train_contigs)
    all_valid_contigs.append(valid_contigs)
    all_test_contigs.append(test_contigs)
    print(f"Finished processing cell type: {cell_type}")

Finished processing cell type: K562
Finished processing cell type: MCF-7
Finished processing cell type: GM12878
Finished processing cell type: HepG2


In [ ]:
all_train_targets_final = np.vstack(all_train_targets)
all_valid_targets_final = np.vstack(all_valid_targets)
all_test_targets_final = np.vstack(all_test_targets)
print(all_train_targets_final.shape)

all_train_contigs_final = pd.concat(all_train_contigs, axis=0)
all_valid_contigs_final = pd.concat(all_valid_contigs, axis=0)
all_test_contigs_final = pd.concat(all_test_contigs, axis=0)

(63944, 1024, 46)


In [ ]:
## Shuffle training data; set the random seed for reproducibility
np.random.seed(1401)
perm = np.random.permutation(all_train_targets_final.shape[0])
all_train_targets_final = all_train_targets_final[perm]
all_train_contigs_final = all_train_contigs_final.iloc[perm]

In [8]:
## Save targets to an H5 file
import h5py
outpath = "/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed/4-cells/data"
h5_path = f"/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed/4-cells/data/all_4cells_chip_target.h5"
with h5py.File(h5_path, 'w') as f:
    f.create_dataset('train_target', 
                     data=all_train_targets_final,
                     dtype=np.float16,
                     )
    f.create_dataset('valid_target', 
                     data=all_valid_targets_final,
                     dtype=np.float16,
                     )
    f.create_dataset('test_target', 
                     data=all_test_targets_final,
                     dtype=np.float16,
                     )
f.close()
print("H5 file saved!")

## Save contigs to an NPZ file
contig_path = f"/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed/4-cells/data/all_4cells_chip_contigs.npz"
np.savez(contig_path,
    train_contigs=all_train_contigs_final.values,
    valid_contigs=all_valid_contigs_final.values,
    test_contigs=all_test_contigs_final.values,
)
print("Contigs file saved!")

H5 file saved!\nContigs file saved!\n

## Signals present in at least two types of cells

In [10]:
import os
import pandas as pd
import numpy as np

df = pd.read_csv("/home/suncz/work/s02/Encode_epigenome/results/preprocessed/[241209]target_human.txt", sep="\t")
df.head()
df = df.query('notes.str.contains("K562|MCF-7|GM12878|HepG2", na=False)')


factor_celltype_count = df.groupby('factors')['notes'].nunique()
factors_in_multiple_celltypes = factor_celltype_count[factor_celltype_count >= 2]

factor_dict = {}
for index,name in enumerate(factors_in_multiple_celltypes.index.tolist()):
    factor_dict[name] = index

# Method 2: if you have the full factors_in_multiple_celltypes index
factors_order = factors_in_multiple_celltypes.index.tolist()

# Create a mapping dict to assign an order index to each factor
factor_position = {factor: i for i, factor in enumerate(factors_order)}

# Add a sort column
df['sort_order'] = df['factors'].map(factor_position)

# Sort by the sort column
df_sorted = df.sort_values('sort_order').drop('sort_order', axis=1)

df_sorted_unique = df_sorted.drop_duplicates(subset=['factors', 'notes'], keep='first')

In [14]:
sequence_bed = pd.read_csv("/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed/IMR-90_only/1m_akita_sequences.bed", sep="\t", header=None)
sequence_bed.columns = ['chrom', 'start', 'end', 'name']
train_index = sequence_bed[~sequence_bed["chrom"].isin(['chr2', 'chr10', 'chr21'])]
valid_index = sequence_bed[sequence_bed["chrom"].isin(['chr2'])]
test_index = sequence_bed[sequence_bed["chrom"].isin(['chr10', 'chr21'])]

In [34]:
BASE_DIR = "/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed"
source_dir = "/home/suncz/work/s02/Encode_epigenome/results/preprocessed/1m-akita/seqs_cov"

all_train_targets = []
all_valid_targets = []
all_test_targets = []

all_train_contigs = []
all_valid_contigs = []
all_test_contigs = []

all_indexes = []

for cell_type in ["K562", "MCF-7", "GM12878", "HepG2"]:
    df_selected = df_sorted_unique[(df_sorted_unique["notes"] == cell_type) & (df_sorted_unique["factors"].isin(factors_in_multiple_celltypes.index))]
    nums_ls = []
    index_ls = []
    for name in df_selected.iterrows():
        line = name[1]
        tfs = line['factors']
        nums = factor_dict[tfs]
        nums_ls.append(nums)
        index_ls.append(line['index'])
    mask = np.zeros(len(factor_dict), dtype=bool)
    mask[nums_ls] = True
    all_indexes.append(mask)
    target_ls = []
    for index in index_ls:
        source_h5 = os.path.join(source_dir, f"{index}.h5.npy")
        target = np.load(source_h5, allow_pickle=True)
        target_ls.append(target)
    targets = np.stack(target_ls,axis=-1)

    full_targets = np.zeros(
        (targets.shape[0], targets.shape[1], len(factor_dict)),
        dtype=targets.dtype
    )

    # Fill in TFs present in the current cell type
    for local_idx, global_idx in enumerate(nums_ls):
        full_targets[:, :, global_idx] = targets[:, :, local_idx]

    all_train_targets.append(full_targets[train_index.index,:,:])
    all_valid_targets.append(full_targets[valid_index.index,:,:])
    all_test_targets.append(full_targets[test_index.index,:,:])
    train_contigs = train_index.copy()
    train_contigs['cell_type'] = cell_type
    valid_contigs = valid_index.copy()
    valid_contigs['cell_type'] = cell_type
    test_contigs = test_index.copy()
    test_contigs['cell_type'] = cell_type
    all_train_contigs.append(train_contigs)
    all_valid_contigs.append(valid_contigs)
    all_test_contigs.append(test_contigs)
    print(f"Finished processing cell type: {cell_type}")

Finished processing cell type: K562
Finished processing cell type: MCF-7
Finished processing cell type: GM12878
Finished processing cell type: HepG2


In [50]:
all_train_targets_final = np.vstack(all_train_targets)
all_valid_targets_final = np.vstack(all_valid_targets)
all_test_targets_final = np.vstack(all_test_targets)
print(all_train_targets_final.shape)

all_train_contigs_final = pd.concat(all_train_contigs, axis=0)
all_valid_contigs_final = pd.concat(all_valid_contigs, axis=0)
all_test_contigs_final = pd.concat(all_test_contigs, axis=0)

(63944, 1024, 208)


In [51]:
## Shuffle training data; set the random seed for reproducibility
np.random.seed(1401)
perm = np.random.permutation(all_train_targets_final.shape[0])
all_train_targets_final = all_train_targets_final[perm]
all_train_contigs_final = all_train_contigs_final.iloc[perm]

In [57]:
all_indexes_dict = {}
for i, cell_type in enumerate(["K562", "MCF-7", "GM12878", "HepG2"]):
    all_indexes_dict[cell_type] = all_indexes[i]

In [58]:
## Save targets to an H5 file
import h5py
outpath = "/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed/EPCOT_strategy/h5"
h5_path = f"/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed/EPCOT_strategy/h5/cell_2_chip_target.h5"
with h5py.File(h5_path, 'w') as f:
    f.create_dataset('train_target', 
                     data=all_train_targets_final,
                     dtype=np.float16,
                     )
    f.create_dataset('valid_target', 
                     data=all_valid_targets_final,
                     dtype=np.float16,
                     )
    f.create_dataset('test_target', 
                     data=all_test_targets_final,
                     dtype=np.float16,
                     )
f.close()
print("H5 file saved!")

## Save contigs to an NPZ file
contig_path = f"/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed/EPCOT_strategy/h5/cell_2_chip_contigs.npz"
np.savez(contig_path,
    train_contigs=all_train_contigs_final.values,
    valid_contigs=all_valid_contigs_final.values,
    test_contigs=all_test_contigs_final.values,
)
print("Contigs file saved!")
## Save indexes_dict
import pickle
indexes_path = f"/home/suncz/work/s02/Encode_epigenome/results/benchmark/EPCOT/dataset/succeed/EPCOT_strategy/h5/cell_2_chip_indexes_dict.pkl"
with open(indexes_path, 'wb') as f:
    pickle.dump(all_indexes_dict, f)
print("Indexes dict saved!")

H5 file saved!\nContigs file saved!\nIndexes dict saved!\n